# SE446 Big Data Engineering — Milestone 2
## Chicago Crime Analytics with Spark + MLlib

**Group:** ChicagoPD

| Name                    | Student ID | M2 Tasks                                            |
|-------------------------|------------|-----------------------------------------------------|
| Ahmad Fares Mzayek      | 230695     | Tasks 9–11 (Deployment) + repo orchestration        |
| Faisal Hajj Khalil      | 230023     | Tasks 1–2 (DataFrame + SQL analytics)               |
| Tanzim Alam             | 220693     | Tasks 3–4 (Trends + arrest rate analysis)           |
| Mohammad Ghassan Hussen | 230367     | Tasks 5–6 (Feature engineering + 3-model training)  |
| Bilal Othman            | 230031     | Task 7 (Feature importances + interpretation)       |

---

### Overview

This notebook upgrades the Milestone 1 MapReduce pipeline to in-memory Spark analytics and an end-to-end MLlib pipeline for arrest prediction.

- **Phase A (Tasks 1–4):** Reproduces M1's MapReduce analyses using Spark DataFrames and Spark SQL.
- **Phase B (Tasks 5–7):** Builds, trains, and compares three classifiers (Logistic Regression, Random Forest, GBT) to predict arrest outcomes.
- **Phase C (Tasks 9–11):** Demonstrates execution in three modes — local, YARN client, and `spark-submit` cluster.

### Execution environments

The notebook auto-detects its environment and adapts:

| Mode      | Where             | Data                                       |
|-----------|-------------------|--------------------------------------------|
| **Local** | Laptop            | Generated 10,000-row sample                |
| **Cluster** | YARN on Hadoop  | Full dataset (`hdfs:///data/chicago_crimes.csv`) |


---
## Setup — Environment Detection & SparkSession

The cells below detect whether the notebook is running on a laptop or on the Hadoop cluster, then build a SparkSession with the appropriate configuration. Each task that follows operates on the loaded DataFrame `df`.

In [1]:
# ============================================
# Setup: Environment Detection
# Author: Ahmad Fares Mzayek (ID: 230695)
# ============================================

import os, sys, subprocess

def detect_environment():
    """Detect whether we are on the Hadoop cluster or a local machine."""
    try:
        result = subprocess.run(
            ["hdfs", "dfs", "-test", "-e", "/data/chicago_crimes.csv"],
            capture_output=True, timeout=5
        )
        if result.returncode == 0:
            return "cluster"
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    return "local"

ENV = detect_environment()
print(f"Environment detected: {ENV.upper()}")

if ENV == "cluster":
    print("  -> Using YARN + HDFS (full Chicago Crimes dataset)")
    print("  -> Data: hdfs:///data/chicago_crimes.csv")
else:
    print("  -> Using local mode (generated sample data)")
    print("  -> No cluster needed")

Environment detected: LOCAL
  -> Using local mode (generated sample data)
  -> No cluster needed


In [2]:
# ============================================
# Setup: Build SparkSession
# Author: Ahmad Fares Mzayek (ID: 230695)
# ============================================

from pyspark.sql import SparkSession

if ENV == "cluster":
    spark = SparkSession.builder \
        .appName("SE446_M2_ChicagoPD") \
        .master("yarn") \
        .config("spark.sql.shuffle.partitions", "8") \
        .getOrCreate()
else:
    spark = SparkSession.builder \
        .appName("SE446_M2_ChicagoPD_Local") \
        .master("local[*]") \
        .config("spark.sql.shuffle.partitions", "4") \
        .config("spark.driver.memory", "2g") \
        .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Master:        {spark.sparkContext.master}")
print(f"Environment:   {ENV}")

26/05/18 20:36:06 WARN Utils: Your hostname, Ahmads-MacBook-Pro-816.local resolves to a loopback address: 127.0.0.1; using 10.10.188.126 instead (on interface en0)
26/05/18 20:36:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/18 20:36:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.4
Master:        local[*]
Environment:   local


In [3]:
# ============================================
# Setup: Load Data
# Author: Ahmad Fares Mzayek (ID: 230695)
# ============================================
#
# In CLUSTER mode, loads the full chicago_crimes.csv from HDFS (7M+ rows).
# In LOCAL mode, generates a 10,000-row synthetic sample with realistic
# arrest-rate patterns per crime type so all downstream tasks work the same.

from pyspark.sql.functions import col, hour, to_timestamp, year, when
from pyspark.sql import Row
import random

if ENV == "cluster":
    # ---- CLUSTER MODE: read full dataset from HDFS ----
    df = spark.read.csv(
        "hdfs:///data/chicago_crimes.csv",
        header=True, inferSchema=True
    )
    # Add an Hour column extracted from the Date string
    df = df.withColumn(
        "Hour", hour(to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a"))
    )
else:
    # ---- LOCAL MODE: generate a 10K-row synthetic sample ----
    random.seed(42)

    crime_profiles = {
        "NARCOTICS":           0.85,
        "PROSTITUTION":        0.80,
        "WEAPONS VIOLATION":   0.60,
        "BATTERY":             0.30,
        "ASSAULT":             0.25,
        "ROBBERY":             0.15,
        "THEFT":               0.10,
        "BURGLARY":            0.08,
        "MOTOR VEHICLE THEFT": 0.06,
        "CRIMINAL DAMAGE":     0.05,
    }
    locations = [
        "STREET", "RESIDENCE", "APARTMENT", "SIDEWALK",
        "PARKING LOT", "ALLEY", "SCHOOL", "RESTAURANT",
        "GAS STATION", "SMALL RETAIL STORE",
    ]
    districts = list(range(1, 26))
    years     = list(range(2001, 2026))

    def gen_row(i):
        crime    = random.choice(list(crime_profiles.keys()))
        base     = crime_profiles[crime]
        district = random.choice(districts)
        yr       = random.choices(years, weights=[max(1, 30 - (y - 2001)) for y in years])[0]
        hr       = random.randint(0, 23)
        location = random.choice(locations)
        domestic = random.random() < 0.15
        arrest_p = base + (0.20 if domestic else 0)
        if 2 <= hr <= 5:
            arrest_p -= 0.10
        arrest_p = max(0.01, min(0.99, arrest_p))
        arrest   = random.random() < arrest_p
        return Row(
            ID=i,
            Date=f"{random.randint(1,12):02d}/{random.randint(1,28):02d}/{yr} "
                 f"{(hr % 12) or 12:02d}:{random.randint(0,59):02d}:00 "
                 f"{'AM' if hr < 12 else 'PM'}",
            **{"Primary Type": crime},
            **{"Location Description": location},
            Arrest=arrest,
            Domestic=domestic,
            District=district,
            Year=yr,
            Hour=hr,
        )

    rows = [gen_row(i) for i in range(10000)]
    df   = spark.createDataFrame(rows)

print(f"Dataset loaded: {df.count():,} rows")
df.printSchema()
df.show(5, truncate=False)

Dataset loaded: 10,000 rows
root
 |-- ID: long (nullable = true)
 |-- Date: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- District: long (nullable = true)
 |-- Year: long (nullable = true)
 |-- Hour: long (nullable = true)

+---+----------------------+-------------------+--------------------+------+--------+--------+----+----+
|ID |Date                  |Primary Type       |Location Description|Arrest|Domestic|District|Year|Hour|
+---+----------------------+-------------------+--------------------+------+--------+--------+----+----+
|0  |12/18/2015 07:05:00 AM|PROSTITUTION       |SIDEWALK            |true  |true    |1       |2015|7   |
|1  |09/07/2001 02:45:00 AM|CRIMINAL DAMAGE    |SIDEWALK            |false |false   |14      |2001|2   |
|2  |03/23/2004 06:27:00 PM|MOTOR VEHICLE THEFT|PARKING LOT         |true  |false   |14    

---
# Phase A — Spark DataFrame Analytics

These four tasks mirror the M1 MapReduce analyses, re-implemented in Spark DataFrames and Spark SQL. Results should match M1 exactly when run against the same dataset on the cluster.

### Task 1: Crime Type Distribution (Spark DataFrame)
**Author: Faisal Hajj Khalil (ID: 230023)**

Reproduce M1 Task 2 (crime type distribution) using the Spark DataFrame API. Show the top 10 crime types by count. The M1 MapReduce result for this analysis is included for side-by-side comparison.

In [4]:
# ============================================
# Task 1: Crime Type Distribution (DataFrame)
# Author: Faisal Hajj Khalil (ID: 230023)
# ============================================
 
from pyspark.sql.functions import col
 
print("=" * 60)
print("Task 1: Top 10 Crime Types (Spark DataFrame)")
print("=" * 60)
 		
crime_counts = (
    df.groupBy("Primary Type")
      .count()
      .orderBy(col("count").desc())
)
 
crime_counts.show(10, truncate=False)
 
# M1 vs M2 comparison note
print("M1 MapReduce result for comparison: THEFT was the top crime type")
print("(162,688 occurrences on the full HDFS dataset). When this notebook")
print("runs in cluster mode, the Spark numbers should match exactly.")



Task 1: Top 10 Crime Types (Spark DataFrame)
+-------------------+-----+
|Primary Type       |count|
+-------------------+-----+
|BATTERY            |1046 |
|THEFT              |1035 |
|WEAPONS VIOLATION  |1029 |
|PROSTITUTION       |1014 |
|CRIMINAL DAMAGE    |1009 |
|MOTOR VEHICLE THEFT|996  |
|BURGLARY           |993  |
|ASSAULT            |984  |
|ROBBERY            |966  |
|NARCOTICS          |928  |
+-------------------+-----+

M1 MapReduce result for comparison: THEFT was the top crime type
(162,688 occurrences on the full HDFS dataset). When this notebook
runs in cluster mode, the Spark numbers should match exactly.


### Task 2: Location Hotspots (Spark SQL)
**Author: Faisal Hajj Khalil (ID: 230023)**

Reproduce M1 Task 3 (location hotspots) using **Spark SQL** (not the DataFrame API) to demonstrate SQL-on-Spark. Show the top 10 location descriptions by count.

In [5]:
# ============================================
# Task 2: Location Hotspots (Spark SQL)
# Author: Faisal Hajj Khalil (ID: 230023)
# ============================================
 
print("=" * 60)
print("Task 2: Top 10 Location Hotspots (Spark SQL)")
print("=" * 60)
 
df.createOrReplaceTempView("crimes")
 
location_hotspots = spark.sql("""
    SELECT `Location Description`, COUNT(*) AS total
    FROM crimes
    GROUP BY `Location Description`
    ORDER BY total DESC
    LIMIT 10
""")
 
location_hotspots.show(truncate=False)
 
# M1 vs M2 comparison note
print("M1 MapReduce result for comparison: STREET was the top location")
print("(245,437 occurrences on the full HDFS dataset). When this notebook")
print("runs in cluster mode, the Spark numbers should match exactly.")


Task 2: Top 10 Location Hotspots (Spark SQL)
+--------------------+-----+
|Location Description|total|
+--------------------+-----+
|RESTAURANT          |1053 |
|SIDEWALK            |1018 |
|APARTMENT           |1017 |
|GAS STATION         |1013 |
|SCHOOL              |1012 |
|RESIDENCE           |1007 |
|STREET              |989  |
|SMALL RETAIL STORE  |979  |
|PARKING LOT         |969  |
|ALLEY               |943  |
+--------------------+-----+

M1 MapReduce result for comparison: STREET was the top location
(245,437 occurrences on the full HDFS dataset). When this notebook
runs in cluster mode, the Spark numbers should match exactly.


### Task 3: Crime Trend Over Years (DataFrame + Visualization)
**Author: Tanzim Alam (ID: 220693)**

Reproduce M1 Task 4 (yearly crime trends) using the DataFrame API. On local mode, render a matplotlib line chart. On cluster mode, print the table (matplotlib may not be installed on workers).

In [ ]:
# ============================================
# Task 3: Crime Trend Over Years
# Author: Tanzim Alam (ID: 220693)
# ============================================

# TODO (Tanzim): groupBy Year, count, orderBy Year.
# Collect to pandas and plot with matplotlib on local;
# print the table on cluster. Save the chart to
# m2/output/figures/crime_trend.png if matplotlib is available.


### Task 4: Arrest Rate Analysis (DataFrame)
**Author: Tanzim Alam (ID: 220693)**

Reproduce M1 Task 5 (arrest analysis) and extend it with a per-crime-type breakdown. Show: (1) overall arrest rate, (2) arrest rate per crime type for the top 10 types. Interpret which types have the highest and lowest arrest rates.

In [ ]:
# ============================================
# Task 4: Arrest Rate Analysis
# Author: Tanzim Alam (ID: 220693)
# ============================================

# TODO (Tanzim): cast Arrest to integer for averaging.
# Compute overall mean. Then group by Primary Type, compute
# count + mean(arrest_int), order by count desc, show top 10.
# Write a short interpretation paragraph below.


**Interpretation (Tanzim):**

*[Fill in once cells above are executed — which crime types have the highest/lowest arrest rates and why.]*

---
# Phase B — Spark MLlib Arrest Prediction

An end-to-end ML pipeline that predicts whether a crime will result in an arrest, using the same Pipeline pattern from the W09B lab.

**Features:** `District`, `crime_index` (from `Primary Type`), `Hour`, `domestic_index` (from `Domestic`)  
**Label:** `Arrest` (cast to integer)

### Task 5: Feature Engineering Pipeline
**Author: Mohammad Ghassan Hussen (ID: 230367)**

Build a Spark ML `Pipeline` with:
1. `StringIndexer` for `Primary Type` → `crime_index`
2. `StringIndexer` for `Domestic` (as string) → `domestic_index`
3. `VectorAssembler` combining `[District, crime_index, Hour, domestic_index]` → `features`
4. An 80/20 train/test split with `seed=42`

Show the `features` column for 5 sample rows and explain what each vector position represents.

In [ ]:
# ============================================
# Task 5: Feature Engineering Pipeline
# Author: Mohammad Ghassan Hussen (ID: 230367)
# ============================================

# TODO (Mohammad):
#  1. Cast Arrest -> label (integer), Domestic -> Domestic_str (string).
#  2. Drop nulls in the columns used as features/label.
#  3. Build crime_indexer, domestic_indexer, assembler.
#  4. Apply them manually once to show 5 rows of the features vector.
#  5. randomSplit([0.8, 0.2], seed=42), cache train_df.


**Vector layout (Mohammad):**

*[Fill in once the features column is shown — e.g., position 0 = District, position 1 = crime_index, position 2 = Hour, position 3 = domestic_index.]*

### Task 6: Train and Evaluate Three Models
**Author: Mohammad Ghassan Hussen (ID: 230367)**

Train and evaluate the three required classifiers with the exact hyperparameters from the spec:

| Model               | Hyperparameters                       |
|---------------------|---------------------------------------|
| Logistic Regression | `maxIter=100`, `regParam=0.01`        |
| Random Forest       | `numTrees=100`, `maxDepth=5`          |
| GBT                 | `maxIter=50`, `maxDepth=5`            |

For each model, report: AUC-ROC, Accuracy, F1 Score, Precision, Recall, confusion matrix (TN, FP, FN, TP), training time. Produce a side-by-side comparison table.

In [ ]:
# ============================================
# Task 6: Train and Evaluate Three Models
# Author: Mohammad Ghassan Hussen (ID: 230367)
# ============================================

# TODO (Mohammad):
#  1. Build three Pipelines, one per model, reusing the indexers + assembler.
#  2. Train each, timing with time.time().
#  3. Use BinaryClassificationEvaluator for AUC-ROC.
#  4. Use MulticlassClassificationEvaluator for accuracy/f1/precision/recall.
#  5. Compute confusion matrix from prediction.groupBy('label','prediction').count().
#  6. Print one side-by-side comparison table at the end.
#  7. Store the trained RF model in a variable named `model_rf` for Task 7.


### Task 7: Feature Importances & Interpretation
**Author: Bilal Othman (ID: 230031)**

Extract and display the feature importances from the Random Forest model (`model_rf` from Task 6). Then answer in prose:

1. Which feature is most important? Does this match the arrest-rate analysis from Task 4?
2. Why does Logistic Regression perform worse than tree-based models on this data?

In [ ]:
# ============================================
# Task 7: Feature Importances & Interpretation
# Author: Bilal Othman (ID: 230031)
# ============================================

# TODO (Bilal):
#  1. Pull the RandomForest stage out of model_rf.stages.
#  2. Pair its .featureImportances with the assembler input column names.
#  3. Print them in a clean table sorted descending.
#  4. Either render a matplotlib bar chart on local, or print ASCII bars.


**Interpretation (Bilal):**

**1. Most important feature & match with Task 4:**

*[Fill in once feature importances are computed.]*

**2. Why Logistic Regression underperforms tree models here:**

*[Fill in once Task 6 results are visible — likely centered on LR's linearity assumption vs. the non-linear interactions tree models capture, especially between Primary Type and Domestic.]*

---
# Phase C — Deployment Modes

Evidence of running this notebook in three modes is captured outside the notebook:

| Task | Mode                          | Evidence location                                      |
|------|-------------------------------|--------------------------------------------------------|
| 9    | Local (`local[*]`)            | `m2/output/screenshots/task9_local.png`               |
| 10   | YARN client (`--master yarn`) | `m2/output/screenshots/task10_yarn_client.png`        |
| 11   | `spark-submit` cluster mode   | `m2/output/spark_submit/run.log` + `m2_spark_ml.py`   |

See the README for the full deployment write-up.

---
## Cleanup

In [ ]:
spark.stop()
print("SparkSession stopped.")